In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
nltk.data.path.clear()
nltk.download('punkt')

In [ ]:
!pip install keybert

In [ ]:
from keybert import KeyBERT
from collections import Counter

In [ ]:
df = pd.read_csv('/content/training_data_binary.csv')

In [ ]:
df.head(5)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_prompt(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return ''.join(text)

In [ ]:
df = pd.read_csv("/content/jailbreak_prompts_DAN.csv")
jailbreak_prompts = df['prompt'].tolist()

In [ ]:
df['preprocessed_prompt'] = df['prompt'].apply(preprocess_prompt)

In [ ]:
jailbreak_prompts = df['preprocessed_prompt'].tolist()

In [ ]:
kw_model = KeyBERT(model='all-MiniLM-L6-v2')

In [ ]:
from tqdm import tqdm

def extract_keywords(prompts, top_n=3):
    all_keywords = []
    for prompt in tqdm(prompts, desc="Extracting keywords"):
        keywords = kw_model.extract_keywords(prompt, top_n=top_n, stop_words='english')
        all_keywords.extend([kw[0] for kw in keywords])
    return Counter(all_keywords)

In [ ]:
jailbreak_keywords = extract_keywords(jailbreak_prompts, top_n=3)

In [ ]:
df_keywords = pd.DataFrame({
    "Jailbreak": pd.Series(jailbreak_keywords)
}).fillna(0).astype(int)

In [ ]:
!pip install squarify

In [ ]:
import matplotlib.pyplot as plt
import squarify
import matplotlib.cm as cm
top_n = 20
most_common = jailbreak_keywords.most_common(top_n)
labels = [f'{k}\n{v}' for k, v in most_common]
sizes = [v for _, v in most_common]
norm = plt.Normalize(min(sizes), max(sizes))
colors = [cm.Reds(norm(value)) for value in sizes]
plt.figure(figsize=(12, 6))
squarify.plot(sizes=sizes, label=labels, color=colors, alpha=0.8, text_kwargs={'fontsize':12})
plt.axis('off')
plt.title("Most Common Jailbreak Words", fontsize=16)
plt.show()


In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=800, height=400,
    background_color='white',
    colormap='Reds'
).generate_from_frequencies(jailbreak_keywords)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()


In [ ]:
df2 = pd.read_csv('/content/all_labelled_cleaned.csv')

In [ ]:
df2.head(10)

In [ ]:
!pip install KeyBERT

In [ ]:
import pandas as pd
from collections import Counter, defaultdict
from keybert import KeyBERT
import re
from tqdm import tqdm
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
kw_model = KeyBERT()
stemmer = PorterStemmer()

In [ ]:
def preprocess(text):
    tokens = text.lower().split()
    return ' '.join(tokens)

In [ ]:
def expand_multilabel_df(df):
    expanded_rows = []
    for _, row in df.iterrows():
        labels = [lbl.strip().strip("'\"") for lbl in row['Labels'].split(',')]
        for label in labels:
            expanded_rows.append({'prompt': row['prompt'], 'label': label})
    return pd.DataFrame(expanded_rows)

In [ ]:
def extract_keywords_by_label(df, top_n=5):
    keywords_per_label = {}
    grouped = df.groupby('label')['prompt'].apply(lambda x: ' '.join(x)).to_dict()

    for label, text in tqdm(grouped.items(), desc="Extracting keywords by label"):
        processed_text = preprocess(text)
        keywords = kw_model.extract_keywords(processed_text, top_n=top_n, stop_words='english')
        keywords_per_label[label] = [kw[0] for kw in keywords]

    return keywords_per_label

In [ ]:
df_expanded = expand_multilabel_df(df2)
keywords_table = extract_keywords_by_label(df_expanded, top_n=5)

In [ ]:
keywords_df = pd.DataFrame.from_dict(keywords_table, orient='index', columns=[f"Keyword {i+1}" for i in range(5)])

In [ ]:
keywords_df.head(10)

In [ ]:
def extract_keywords_by_label(df, top_n=5):
    keywords_per_label = {}
    grouped = df.groupby('label')['prompt'].apply(lambda x: ' '.join(x)).to_dict()

    for label, text in tqdm(grouped.items(), desc="Extracting keywords by label"):
        processed_text = preprocess(text)
        keywords = kw_model.extract_keywords(processed_text, top_n=top_n, stop_words='english')
        keywords_per_label[label] = dict(keywords)

    return keywords_per_label

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import squarify
import matplotlib.cm as cm

def plot_all_keyword_visuals(keywords_per_label):
    for label, keywords in keywords_per_label.items():
        print(f"\n📌 Visualizing: {label}")
        keywords = dict(sorted(keywords.items(), key=lambda x: x[1], reverse=True))
        keys = list(keywords.keys())
        values = list(keywords.values())
        plt.figure(figsize=(8, 4))
        plt.bar(keys, values, color='teal')
        plt.title(f"Top Keywords - {label}")
        plt.xticks(rotation=45)
        plt.ylabel("Relevance Score")
        plt.tight_layout()
        plt.show()

        wc = WordCloud(width=600, height=300, background_color="white", colormap="Greens")
        wc.generate_from_frequencies(keywords)
        plt.figure(figsize=(8, 4))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"Word Cloud - {label}")
        plt.tight_layout()
        plt.show()

        norm = plt.Normalize(min(values), max(values))
        colors = [cm.Greens(norm(v)) for v in values]
        labels = [f'{k}\n{round(v, 2)}' for k, v in keywords.items()]
        plt.figure(figsize=(8, 4))
        squarify.plot(sizes=values, label=labels, color=colors, alpha=0.8, text_kwargs={'fontsize':12})
        plt.axis('off')
        plt.title(f"Treemap - {label}")
        plt.tight_layout()
        plt.show()


In [ ]:
keywords_per_label = extract_keywords_by_label(df_expanded, top_n=5)

In [ ]:
plot_all_keyword_visuals(keywords_per_label)